# 10 - RAG Evaluation


## What this notebook is for

This notebook is written as **notes + code**. Read the Markdown cells first, then run the code cells.

The goal is to understand the concept, see the architecture, implement a small example, and then evaluate/improve it.

## Learning objectives

- Build an evaluation dataset.
- Calculate retrieval metrics.
- Understand RAG answer metrics.
- Create an improvement loop based on measurements.

## Core AI concept

Evaluation answers: 'Did we retrieve the right evidence?' and 'Did the answer use it correctly?' Improvement should be evidence-driven. A strong workflow is: baseline → measure → identify bottleneck → change one component → re-measure → keep/regress.

## Architecture / mental model

```text
Dataset
 ↓
Baseline RAG
 ↓
Metrics
 ↓
Failure analysis
 ↓
One targeted change
 ↓
Re-run benchmark
 ↓
Regression decision
```

## Connection to the supplied learning material

Your RAG notebook includes test questions and expected source files. The notes here extend that into a broader benchmark/evaluation framework.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

This notebook is intentionally evaluation-focused. Improvement is moved to Notebook 11.

## Setup

The code below uses environment variables for secrets and model names so the notebook is easier to maintain when providers or model versions change.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

print("Model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBED_MODEL)

### Retrieval metrics

In [ ]:
def recall_at_k(retrieved_ids, relevant_ids, k):
    top = retrieved_ids[:k]
    return int(any(x in relevant_ids for x in top))

def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, item in enumerate(retrieved_ids, start=1):
        if item in relevant_ids:
            return 1.0 / rank
    return 0.0

retrieved = ["d3", "d8", "d2", "d1"]
relevant = {"d1", "d2"}

print("Recall@3:", recall_at_k(retrieved, relevant, 3))
print("MRR:", reciprocal_rank(retrieved, relevant))

### Simple experiment logger

In [ ]:
import pandas as pd

results = pd.DataFrame([
    {"name": "baseline", "recall_at_3": 0.72, "faithfulness": 0.81},
    {"name": "rerank", "recall_at_3": 0.72, "faithfulness": 0.88},
    {"name": "query_rewrite", "recall_at_3": 0.80, "faithfulness": 0.87},
])

results["overall_proxy"] = (
    0.5 * results["recall_at_3"] +
    0.5 * results["faithfulness"]
)
results.sort_values("overall_proxy", ascending=False)

### Failure taxonomy

In [ ]:
failure_taxonomy = {
    "missed_evidence": "Improve retrieval recall/chunking/query transformation.",
    "noisy_context": "Improve ranking, filtering, MMR, reranking.",
    "unsupported_answer": "Improve grounding instructions, citations, model quality.",
    "too_many_tokens": "Reduce k, compress context, improve ranking.",
    "slow_system": "Cache embeddings/retrieval, reduce model/tool calls.",
}
failure_taxonomy

## Evaluation

Use retrieval metrics plus RAG metrics such as Context Precision, Context Recall, Faithfulness, and Response Relevancy. Ragas also documents agent/tool metrics separately, which is useful later.

## Improvement ideas

Build a regression suite. Every improvement must beat or preserve the baseline on important cases, not only on the latest question you happen to like.

## Practice exercises

1. Create 30-question benchmark.
2. Add per-source failure analysis.
3. Track latency and token usage beside quality metrics.

## Evaluation dataset design

A useful RAG evaluation dataset has:

- `user_input`: the question
- `reference`: expected answer, when available
- `reference_contexts`: gold supporting chunks, when available
- retrieved contexts: what the system actually found
- response: what the generator actually answered

You should include easy, normal, ambiguous, out-of-scope, and adversarial questions. A benchmark made only of easy questions is a decorative spreadsheet.

In [ ]:
eval_rows = [
    {
        'user_input': 'What is RAG?',
        'reference': 'RAG combines retrieval with generation.',
        'reference_contexts': ['RAG combines retrieval with generation.'],
    },
    {
        'user_input': 'What is an embedding?',
        'reference': 'An embedding represents text as a numerical vector.',
        'reference_contexts': ['An embedding represents text as a numerical vector.'],
    },
]

for row in eval_rows:
    print(row)

## Core metric meanings

**Recall@K**: did the relevant evidence appear in the top K?

**Precision@K**: how much of the top K is relevant?

**MRR**: how high is the first relevant result?

**NDCG**: how good is the ranking when relevance has multiple levels?

**Context Precision**: are useful chunks ranked high?

**Context Recall**: did retrieval find enough relevant information?

**Faithfulness**: are response claims supported by retrieved context?

**Response Relevancy**: does the response answer the question?

In [ ]:
def precision_at_k(retrieved, relevant, k):
    top = retrieved[:k]
    return sum(x in relevant for x in top) / max(1, len(top))

def recall_at_k(retrieved, relevant, k):
    return int(bool(set(retrieved[:k]) & set(relevant)))

def reciprocal_rank(retrieved, relevant):
    for rank, item in enumerate(retrieved, start=1):
        if item in relevant:
            return 1 / rank
    return 0.0

retrieved = ['noise', 'd2', 'd3']
relevant = {'d2'}
print('Precision@3:', precision_at_k(retrieved, relevant, 3))
print('Recall@3:', recall_at_k(retrieved, relevant, 3))
print('MRR:', reciprocal_rank(retrieved, relevant))

## Current Ragas metric map

The current Ragas documentation lists RAG metrics including Context Precision, Context Recall, Response Relevancy, Faithfulness, Multimodal Faithfulness, and Multimodal Relevance. The same documentation also lists agent/tool metrics, which belong to the later Agentic AI track.

Use Ragas for automated measurement, but keep deterministic application metrics too. One metric should never be the only judge of the system.

## Evaluation report structure

For each experiment, store:

```text
benchmark_version
code_version
model_version
chunking_config
retrieval_config
reranker_config
retrieval_metrics
answer_metrics
latency
tokens/cost
known_failures
```

This makes regressions explainable.

## Final checklist

- [ ] I can explain the concept in simple words.
- [ ] I can describe where it fits in a GenAI system.
- [ ] I can run and modify the code.
- [ ] I know at least one way to evaluate it.
- [ ] I know what to change when it fails.

## Research / reference notes


- LangChain Python reference: https://reference.langchain.com/python/langchain/overview
- LangGraph Python reference: https://reference.langchain.com/python/langgraph/overview
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- OpenAI function calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API: https://developers.openai.com/api/reference/cli/resources/responses/methods/create

- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/